
# TITAN smoke test on RunPod (remote-only)
- PoC/v2용. 로컬에서 SSH로 RunPod `/workspace`에 접속해 **레벨1 타일링 → CONCH v1.5 patch → TITAN slide embedding → 썸네일/히트맵**까지 원격 실행.
- v1 `gigapath_runpod_modeling.ipynb` 스타일의 SSH/rsync/venv 관리 방식을 동일하게 사용.
- 실행 전: RunPod 노드 가동, SSH 키/호스트/포트 값 업데이트, 원격 `/workspace/PoC/v2`에 리포가 존재하도록 동기화.


In [1]:

import os, shlex, subprocess, json, time
from pathlib import Path

# --- SSH 설정 (필요에 맞게 수정) ---
SSH_HOST_GATEWAY = 'vvy5ke98vgtmim-64411852@ssh.runpod.io'
SSH_HOST_DIRECT = 'root@69.30.85.100'
SSH_PORT_DIRECT = 22027
SSH_KEY = '~/.ssh/runpod_peter'
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
SSH_EXTRA_OPTS = ''  # 예: '-o StrictHostKeyChecking=no'

# --- 원격 경로 설정 ---
REMOTE_BASE = '/workspace/PoC/v2'
REMOTE_CODE = f"{REMOTE_BASE}/TITAN"
REMOTE_OUTPUT = f"{REMOTE_BASE}/output"
REMOTE_VENV = f"{REMOTE_BASE}/.venv"
REMOTE_DATA_ROOT = '/workspace/data'  # SVS 위치 상위

# --- 실행 파라미터 ---
RUN_REMOTE_SETUP = True   # venv 생성+패키지 설치
RUN_EXTRACT = False       # True로 두면 바로 임베딩 실행

# 테스트할 SVS (원격 경로). 예시는 빈 리스트 → 필요시 채우세요.
WSI_LIST = [
    # f"{REMOTE_DATA_ROOT}/raw/S23-00000#1###0.svs",
]
LEVEL = 1
PATCH_SIZE_LV = 256   # level1 patch 크기 (20x → level0 512)
BATCH_SIZE = 32
BG_WHITE_RATIO = 0.80
MAX_TILES = None       # 숫자로 제한하면 스모크, None이면 전체

# Torch/cu 버전은 RunPod 이미지에 맞춰 필요시 수정
TORCH_INDEX_URL = os.environ.get('TORCH_INDEX_URL', 'https://download.pytorch.org/whl/cu121')
TORCH_VERSION = '2.4.1'
TORCHVISION_VERSION = '0.19.1'

print('Remote base:', REMOTE_BASE)


Remote base: /workspace/PoC/v2


In [2]:

# SSH 유틸

def _ssh_parts(host=None, port=None):
    host = host or (SSH_HOST_DIRECT if USE_DIRECT_FOR_SSH else SSH_HOST_GATEWAY)
    port = port or (SSH_PORT_DIRECT if USE_DIRECT_FOR_SSH else None)
    parts = ['ssh', '-i', SSH_KEY]
    if port:
        parts += ['-p', str(port)]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    parts.append(host)
    return parts


def run_ssh(cmd, check=True, env=None):
    env_prefix = ''
    if env:
        env_prefix = ' '.join([f"{k}={shlex.quote(str(v))}" for k,v in env.items()]) + ' '
    full_cmd = ' '.join(_ssh_parts() + [shlex.quote(env_prefix + cmd)])
    print('[ssh]', full_cmd)
    result = subprocess.run(full_cmd, shell=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"SSH command failed: {cmd}")
    return result


def run_local(cmd, check=True):
    print('[local]', cmd)
    result = subprocess.run(cmd, shell=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Local command failed: {cmd}")
    return result


def run_rsync(src, dst, to_remote=True, opts='-azP'):
    host = SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY
    port = SSH_PORT_DIRECT if USE_DIRECT_FOR_RSYNC else None
    port_opt = f"-e 'ssh -i {SSH_KEY} -p {port}'" if port else f"-e 'ssh -i {SSH_KEY}'"
    if to_remote:
        cmd = f"rsync {opts} {port_opt} {src} {host}:{dst}"
    else:
        cmd = f"rsync {opts} {port_opt} {host}:{src} {dst}"
    run_local(cmd)



## 1) 원격 venv/패키지 설치
- v1 runpod 노트북과 동일하게 `/workspace/PoC/v2/.venv` 사용.
- Torch/cu 인덱스는 RunPod 이미지에 맞춰 `TORCH_INDEX_URL`로 조정 가능.


In [3]:

if RUN_REMOTE_SETUP:
    setup_script = f"""
    set -e
    mkdir -p {REMOTE_BASE}
    cd {REMOTE_BASE}
    if [ ! -d {REMOTE_VENV} ]; then
        python -m venv {REMOTE_VENV}
    fi
    source {REMOTE_VENV}/bin/activate
    pip install --upgrade pip
    pip install torch=={TORCH_VERSION} torchvision=={TORCHVISION_VERSION} --index-url {TORCH_INDEX_URL}
    pip install timm==1.0.3 einops==0.6.1 einops-exts==0.0.4 tqdm==4.66.6         h5py==3.8.0 transformers==4.46.0 pandas==2.2.3 scikit-learn==1.5.2         matplotlib seaborn opencv-python openslide-python huggingface_hub==0.26.5
    pip install -e {REMOTE_CODE}
    """
    run_ssh(setup_script)
else:
    print('RUN_REMOTE_SETUP=False → 설치 단계 건너뜀')


[ssh] ssh -i ~/.ssh/runpod_peter -p 22027 root@69.30.85.100 '
    set -e
    mkdir -p /workspace/PoC/v2
    cd /workspace/PoC/v2
    if [ ! -d /workspace/PoC/v2/.venv ]; then
        python -m venv /workspace/PoC/v2/.venv
    fi
    source /workspace/PoC/v2/.venv/bin/activate
    pip install --upgrade pip
    pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
    pip install timm==1.0.3 einops==0.6.1 einops-exts==0.0.4 tqdm==4.66.6         h5py==3.8.0 transformers==4.46.0 pandas==2.2.3 scikit-learn==1.5.2         matplotlib seaborn opencv-python openslide-python huggingface_hub==0.26.5
    pip install -e /workspace/PoC/v2/TITAN
    '
  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━

KeyboardInterrupt: 


## 2) Hugging Face 로그인 (원격 세션)
- TITAN/CONCH 다운로드를 위해 1회 토큰 입력 필요.
- 이미 로그인 상태면 건너뛰어도 됨.


In [ ]:

run_ssh(f"""
source {REMOTE_VENV}/bin/activate
python - <<'PY'
from huggingface_hub import login
print('If already logged in, press Enter to keep existing token.')
login()
PY
""")



## 3) TITAN 임베딩 실행 (원격)
- 레벨1 타일링, 배경 필터링, CONCH patch 추출 → H5 저장(features/coords/patch_size_level0).
- TITAN slide embedding → PT 저장.
- 썸네일/히트맵/coords.csv 함께 저장.
- `WSI_LIST`를 원격 SVS 경로로 채워 `RUN_EXTRACT=True`로 설정 후 실행.


In [ ]:

if not RUN_EXTRACT:
    print('RUN_EXTRACT=False → 실행 안 함. 위에서 True로 바꾸고 다시 실행하세요.')
else:
    if not WSI_LIST:
        raise ValueError('WSI_LIST가 비어 있습니다. 원격 SVS 경로를 지정하세요.')

    wsi_json = json.dumps(WSI_LIST)
    extract_script = f"""
    set -e
    source {REMOTE_VENV}/bin/activate
    python - <<'PY'
import os, csv, json, math
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel
from PIL import Image
import h5py
import openslide
from matplotlib import cm

BASE_DIR = Path('{REMOTE_BASE}')
OUTPUT_ROOT = Path('{REMOTE_OUTPUT}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WSI_LIST = json.loads("""{wsi_json}""")
LEVEL = {LEVEL}
PATCH_SIZE_LV = {PATCH_SIZE_LV}
BATCH_SIZE = {BATCH_SIZE}
BG_WHITE_RATIO = {BG_WHITE_RATIO}
MAX_TILES = {json.dumps(MAX_TILES)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', DEVICE)
print('Slides:', WSI_LIST)

print('Loading TITAN + CONCH v1.5...')
titan = AutoModel.from_pretrained('MahmoodLab/TITAN', trust_remote_code=True)
conch, eval_transform = titan.return_conch()
titan = titan.to(DEVICE).eval()
conch = conch.to(DEVICE).eval()
print('Models ready')


def load_slide(slide_path: str):
    slide = openslide.OpenSlide(slide_path)
    level0_w, level0_h = slide.dimensions
    if LEVEL >= slide.level_count:
        raise ValueError(f"Slide has only {slide.level_count} levels; requested LEVEL={LEVEL}")
    downsample = slide.level_downsamples[LEVEL]
    return slide, level0_w, level0_h, downsample


def compute_grid(level0_w, level0_h, downsample, patch_size_lv):
    stride_lv0 = int(patch_size_lv * downsample)
    xs = range(0, level0_w - stride_lv0 + 1, stride_lv0)
    ys = range(0, level0_h - stride_lv0 + 1, stride_lv0)
    coords_lv0 = [(x, y) for y in ys for x in xs]
    return coords_lv0, stride_lv0


def is_background(pil_img, white_ratio=BG_WHITE_RATIO):
    arr = np.asarray(pil_img).astype(np.uint8)
    if arr.ndim == 3 and arr.shape[2] == 4:
        arr = arr[:, :, :3]
    white = (arr > 220).mean()
    return white > white_ratio


def make_thumbnail(slide, max_dim=2048):
    w, h = slide.dimensions
    scale = max(w, h) / max_dim
    thumb_size = (int(w / scale), int(h / scale))
    return slide.get_thumbnail(thumb_size).convert('RGB')


def conch_batch(patches):
    tensors = [eval_transform(p) for p in patches]
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad(), torch.autocast(DEVICE.type, torch.float16):
        feats = conch(batch)
    return feats.cpu()


def build_heatmap(coords_lv0, scores, thumb, level0_w, level0_h, stride_lv0):
    scores = np.asarray(scores)
    norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-6)
    heat = np.zeros((thumb.size[1], thumb.size[0]), dtype=np.float32)
    for (x0, y0), s in zip(coords_lv0, norm):
        x1 = int((x0 / level0_w) * thumb.size[0])
        y1 = int((y0 / level0_h) * thumb.size[1])
        patch_w = max(1, int((stride_lv0 / level0_w) * thumb.size[0]))
        patch_h = max(1, int((stride_lv0 / level0_h) * thumb.size[1]))
        heat[y1:y1+patch_h, x1:x1+patch_w] = np.maximum(heat[y1:y1+patch_h, x1:x1+patch_w], s)
    cmap = cm.get_cmap('jet')
    heat_rgba = (cmap(heat) * 255).astype(np.uint8)
    return Image.fromarray(heat_rgba).convert('RGBA')


for slide_path in WSI_LIST:
    slide_id = Path(slide_path).stem
    out_dir = OUTPUT_ROOT / slide_id
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"
=== {slide_id} ===")

    slide, level0_w, level0_h, downsample = load_slide(slide_path)
    coords_lv0, stride_lv0 = compute_grid(level0_w, level0_h, downsample, PATCH_SIZE_LV)
    patch_size_lv0 = int(PATCH_SIZE_LV * downsample)
    print(f"Level0: {level0_w}x{level0_h}, level{LEVEL} downsample {downsample}, grid {len(coords_lv0)}")

    features_list, coords_kept, batch, batch_coords = [], [], [], []

    for (x0, y0) in coords_lv0:
        if MAX_TILES is not None and len(coords_kept) >= MAX_TILES:
            break
        tile = slide.read_region((x0, y0), LEVEL, (PATCH_SIZE_LV, PATCH_SIZE_LV)).convert('RGB')
        if is_background(tile):
            continue
        batch.append(tile)
        batch_coords.append((x0, y0))
        if len(batch) == BATCH_SIZE:
            feats = conch_batch(batch)
            features_list.append(feats)
            coords_kept.extend(batch_coords)
            batch, batch_coords = [], []
    if batch:
        feats = conch_batch(batch)
        features_list.append(feats)
        coords_kept.extend(batch_coords)

    if not coords_kept:
        print('No tiles kept; skipped')
        continue

    features = torch.cat(features_list, dim=0)
    coords_tensor = torch.tensor(coords_kept, dtype=torch.int32)
    print('Kept tiles:', features.shape[0])

    # Save patch features
    h5_path = out_dir / f"{slide_id}_features.h5"
    with h5py.File(h5_path, 'w') as f:
        f.create_dataset('features', data=features.numpy(), compression='gzip')
        f.create_dataset('coords', data=coords_tensor.numpy(), compression='gzip')
        f['coords'].attrs['patch_size_level0'] = patch_size_lv0
    with open(out_dir / f"{slide_id}_coords.csv", 'w', newline='') as fp:
        writer = csv.writer(fp)
        writer.writerow(['x_lv0', 'y_lv0'])
        writer.writerows(coords_kept)

    # Slide embedding
    with torch.no_grad(), torch.autocast(DEVICE.type, torch.float16):
        slide_embedding = titan.encode_slide_from_patch_features(
            features.to(DEVICE), coords_tensor.to(DEVICE), patch_size_lv0)
    torch.save(slide_embedding.cpu(), out_dir / f"{slide_id}_titan.pt")

    # Heatmap
    thumb = make_thumbnail(slide)
    scores = F.cosine_similarity(features, slide_embedding.cpu().unsqueeze(0), dim=1).numpy()
    heat_rgba = build_heatmap(coords_kept, scores, thumb, level0_w, level0_h, stride_lv0)
    thumb_rgba = thumb.convert('RGBA')
    overlay = Image.blend(thumb_rgba, heat_rgba, alpha=0.45)
    thumb.save(out_dir / f"{slide_id}_thumbnail.png")
    overlay.save(out_dir / f"{slide_id}_heatmap.png")

    print('Saved:', h5_path, 'and embedding/heatmap files')

print('
Done.')
PY
"""
    run_ssh(extract_script)
